1. Setup & path

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from IPython.display import Image, display

# Asumsi notebook dijalankan dari root repo
ROOT = Path().resolve()
TAB = ROOT / "reports" / "tables"
FIG = ROOT / "reports" / "figures"
ART = ROOT / "artifacts"

TAB, FIG, ART

2. Lihat ringkasan metrik terbaru per dataset dan varian

In [ ]:
# pilih dataset: "tectonic" atau "volcanic"
ds = "volcanic"  # atau "tectonic"

metrics_files = sorted(TAB.glob(f"{ds}_stack_*_metrics.csv"))
metrics_files

3. Load dan display metrik

In [ ]:
# Gabungkan semua metrics.csv untuk dataset terpilih
dfs = [pd.read_csv(p).assign(source=p.name) for p in metrics_files]
metrics = pd.concat(dfs, ignore_index=True)

# Urutkan berdasarkan kinerja (F1, recall, ROC-AUC)
metrics_sorted = metrics.sort_values(
    ["f1", "recall", "roc_auc"],
    ascending=[False, False, False],
)

metrics_sorted.head(20)

4. Ambil model stacking terbaik dan tampilkan Confusion / ROC / PR

In [ ]:
# Pilih hanya baris untuk model stacking (stacking_lr_meta)
stack_rows = metrics_sorted[metrics_sorted["model"] == "stacking_lr_meta"].copy()
if stack_rows.empty:
    raise RuntimeError("Tidak ditemukan baris 'stacking_lr_meta' di metrics.")

best_row = stack_rows.iloc[0]
best_row

In [ ]:
# Ekstrak nama file metrics dan varian (nosmote / smote / smote_tomek / smote_enn)
best_source = best_row["source"]  # contoh: 'volcanic_stack_smote_metrics.csv'
best_source

In [ ]:
# Parse suffix varian dari nama file
# pola: {dataset}_stack_{suffix}_metrics.csv
fname = Path(best_source).name
suffix = fname.replace(f"{ds}_stack_", "").replace("_metrics.csv", "")
suffix

In [ ]:
# Path gambar dari stacking_pipeline.py
cm_png  = FIG / f"{ds}_stack_{suffix}_cm.png"
roc_png = FIG / f"{ds}_stack_{suffix}_roc.png"
pr_png  = FIG / f"{ds}_stack_{suffix}_pr.png"

cm_png, roc_png, pr_png

In [ ]:
# Tampilkan Confusion Matrix (heatmap), ROC, dan PR Curve
if cm_png.exists():
    display(Image(filename=str(cm_png)))
else:
    print("Confusion matrix PNG tidak ditemukan:", cm_png)

if roc_png.exists():
    display(Image(filename=str(roc_png)))
else:
    print("ROC PNG tidak ditemukan:", roc_png)

if pr_png.exists():
    display(Image(filename=str(pr_png)))
else:
    print("PR PNG tidak ditemukan:", pr_png)

5. Periksa threshold dan metadata model terbaik

In [ ]:
# Jika kamu ingin eksplisit pakai varian terbaik di atas:
artifact_path = ART / f"{ds}_stack_{suffix}.joblib"
artifact_path

In [ ]:
art = joblib.load(artifact_path)

thr = float(art.get("decision_threshold", 0.5))
meta = art.get("meta", {})
runtime = art.get("runtime", {})

thr, meta, runtime

6. Tes prediksi cepat pada sampel

In [ ]:
from tsunami_prediction.serve_api import (
    predict_tectonic_stacking,
    predict_volcanic_stacking,
)

In [ ]:
# Contoh input tectonic
X_tect = pd.DataFrame(
    [
        {
            "mag": 7.9,
            "depth": 20,
            "latitude": -3.0,
            "longitude": 100.5,
            "country": "Indonesia",
            "zone": None,                 # opsional; boleh diisi jika ada
            "distance_to_coast_km": 10.0, # jika tidak tahu, bisa 0.0
            "is_subduction_zone": 1,      # 1 jika di zona subduksi, 0 jika tidak / tidak tahu
        }
    ]
)
tect_pred = predict_tectonic_stacking(X_tect)
tect_pred

In [ ]:
# Contoh input volcanic
X_volc = pd.DataFrame(
    [
        {
            "eq": 5.5,
            "elevation": 1200,
            "latitude": -7.9,
            "longitude": 112.3,
            "country": "Indonesia",
            "type": "Caldera",
            "vei": 4,
            "distance_to_coast_km": 5.0,
            "is_subduction_zone": 1,   # 1 jika di zona subduksi
        }
    ]
)
volc_pred = predict_volcanic_stacking(X_volc)
volc_pred

7. Analisis ablation SMOTE dan runtime

In [ ]:
# Ablation SMOTE (nosmote vs smote vs smote_tomek vs smote_enn)
ab = pd.read_csv(TAB / "ablation_smote.csv")
ab_sorted = (
    ab.sort_values(
        ["dataset", "f1", "recall"],
        ascending=[True, False, False],
    )
    .groupby("dataset")
    .head(10)
)
ab_sorted

In [ ]:
# Ringkasan waktu eksekusi per dataset/varian (stack_fit_sec, grid_sec, run_sec)
runtime = pd.read_csv(TAB / "runtime_summary.csv")
runtime

In [ ]:
# Ringkasan model dan hyperparameter:
hyper = pd.read_csv(TAB / "model_hyperparams.csv")
hyper.head(20)